1) Analizar la aplicación Flask básica. Explicar el funcionamiento de app = Flask(name), @app.route("/"), jsonify() y el modo debug=True, tomando como base Prueba_Flask_Routes_01.py.
```python
    app = Flask(__name__)
    
   
   -Esta linea crea una instancia de la clase Flask que actúa como nuestra aplicación web central.
   -El argumento __name__: Es una variable especial de Python que le indica a Flask dónde buscar recursos como archivos estáticos o plantillas HTML.
       

   @app.route("/")
    

    -Es un decorador de Python que asocia una función específica con una dirección URL (punto de acceso).
    -La función vinculada devuelve el texto "Hola", que es lo que el navegador mostrará al usuario.

   jsonify()
    

    -Es una función de Flask que convierte estructuras de datos de Python (como diccionarios o listas) en un formato JSON.

    debug=True

    -Se activa dentro de app.run() y es una herramienta esencial durante la etapa de desarrollo.
    -Cada vez que guardas un cambio en Prueba_Flask_Routes_01.py, el servidor se reinicia solo para aplicar los cambios sin que tengas que cerrarlo y abrirlo manualmente.
        

2) Puntos del 2 al 5:
   ```python
   from flask import Flask, jsonify, request
    import database2 as db
    
    app = Flask(__name__)
    
    # Asegura que la tabla exista al arrancar
    db.crear_tabla()
    
    @app.route("/")
    def index():
        # Mantenemos tu interfaz visual de Bootstrap
        return """
        <!doctype html>
        <html lang="es">
        <head>
            <meta charset="utf-8">
            <meta name="viewport" content="width=device-width, initial-scale=1">
            <title>Flask Sensores Pro</title>
            <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css" rel="stylesheet">
        </head>
        <body class="bg-light">
        <nav class="navbar navbar-dark bg-dark mb-4">
            <div class="container">
                <span class="navbar-brand mb-0 h1">Servidor Flask - Sistema de Lecturas</span>
            </div>
        </nav>
        <div class="container">
            <div class="alert alert-success">Servidor activo en el puerto 5001. Base de datos: lectura_sensores</div>
            <div class="row g-4">
                <div class="col-md-6">
                    <div class="card shadow-sm">
                        <div class="card-header bg-primary text-white">Login por POST</div>
                        <div class="card-body">
                            <form action="/login" method="post">
                                <div class="mb-3">
                                    <label class="form-label">Nombre</label>
                                    <input type="text" name="nombre" class="form-control" placeholder="javier">
                                </div>
                                <div class="mb-3">
                                    <label class="form-label">Clave</label>
                                    <input type="password" name="clave" class="form-control" placeholder="abc">
                                </div>
                                <button type="submit" class="btn btn-primary">Ingresar</button>
                            </form>
                        </div>
                    </div>
                </div>
                <div class="col-md-6">
                    <div class="card shadow-sm">
                        <div class="card-header bg-success text-white">Accesos rápidos</div>
                        <div class="card-body">
                            <a href="/login_get?nombre=javier&clave=abc" class="btn btn-outline-primary mb-2 w-100">Probar Login GET</a>
                            <a href="/sensors" class="btn btn-outline-success mb-2 w-100">Ver JSON (Todas las lecturas)</a>
                        </div>
                    </div>
                </div>
            </div>
        </div>
        </body>
        </html>
        """
    
    # --- RUTAS DE LOGIN (Requisito: Comparar GET vs POST) ---
    
    @app.route("/login", methods=["POST"])
    def login_post():
        # Uso de request.form para datos ocultos en el cuerpo
        nombre = request.form.get("nombre")
        clave = request.form.get("clave")
        return jsonify({"metodo": "POST", "usuario": nombre, "info": "Clave enviada de forma segura en el body"}), 200
    
    @app.route("/login_get", methods=["GET"])
    def login_get():
        # Uso de request.args para datos visibles en la URL
        nombre = request.args.get("nombre")
        clave = request.args.get("clave")
        return jsonify({"metodo": "GET", "usuario": nombre, "clave_expuesta": clave}), 200
    
    # --- CRUD DE SENSORES (Con todos los campos nuevos) ---
    
    @app.route("/sensors", methods=["GET"])
    def listar_sensores():
        sensores = db.obtener_sensores()
        return jsonify({"sensors": sensores}), 200
    
    @app.route("/sensors", methods=["POST"])
    def crear_sensor():
        if not request.is_json:
            return jsonify({"error": "Debe enviar JSON"}), 415
    
        data = request.get_json()
        
        # Mapeamos todos los campos requeridos
        nuevo_sensor = {
            "co2": data.get("co2"),
            "temp": data.get("temp"),
            "hum": data.get("hum"),
            "fecha": data.get("fecha"),
            "lugar": data.get("lugar"),
            "altura": data.get("altura"),
            "presion": data.get("presion"),
            "presion_nm": data.get("presion_nm"),
            "temp_ext": data.get("temp_ext")
        }
    
        id_generado = db.crear_sensor(nuevo_sensor)
        nuevo_sensor["id"] = id_generado
    
        return jsonify({"mensaje": "Lectura creada", "sensor": nuevo_sensor}), 201
    
    @app.route("/sensors/<int:id>", methods=["GET"])
    def consultar_sensor(id):
        sensor = db.obtener_sensor_por_id(id)
        if not sensor:
            return jsonify({"error": "Sensor no encontrado"}), 404
        return jsonify({"sensor": sensor}), 200
    
    @app.route("/sensors/<int:id>", methods=["PUT"])
    def modificar_sensor(id):
        if not request.is_json:
            return jsonify({"error": "Debe enviar JSON"}), 415
    
        sensor_existente = db.obtener_sensor_por_id(id)
        if not sensor_existente:
            return jsonify({"error": "Sensor no encontrado"}), 404
    
        data = request.get_json()
    
        # Actualizamos campos permitiendo mantener los valores viejos si no se envían nuevos
        sensor_actualizado = {
            "co2": data.get("co2", sensor_existente["co2"]),
            "temp": data.get("temp", sensor_existente["temp"]),
            "hum": data.get("hum", sensor_existente["hum"]),
            "fecha": data.get("fecha", sensor_existente["fecha"]),
            "lugar": data.get("lugar", sensor_existente["lugar"]),
            "altura": data.get("altura", sensor_existente["altura"]),
            "presion": data.get("presion", sensor_existente["presion"]),
            "presion_nm": data.get("presion_nm", sensor_existente["presion_nm"]),
            "temp_ext": data.get("temp_ext", sensor_existente["temp_ext"])
        }
    
        db.actualizar_sensor(id, sensor_actualizado)
        sensor_actualizado["id"] = id
        
        return jsonify({"mensaje": "Actualizado correctamente", "sensor": sensor_actualizado}), 200
    
    @app.route("/sensors/<int:id>", methods=["DELETE"])
    def eliminar_sensor(id):
        if db.eliminar_sensor(id):
            return jsonify({"mensaje": "Sensor eliminado", "result": True}), 200
        return jsonify({"error": "No se pudo eliminar o no existe"}), 404
    
    if __name__ == "__main__":
        app.run(host="0.0.0.0", port=5001, debug=True)
```python
-Funciones base de datos:

    import sqlite3
    
    DB_NAME = "sensores.db"
    
    def conectar():
        """Establece conexión con la base de datos y permite acceso por nombres de columna."""
        conn = sqlite3.connect(DB_NAME, timeout=10)
        # Importante: Permite manejar las filas como diccionarios
        conn.row_factory = sqlite3.Row
        return conn
    
    def crear_tabla():
        """Crea la tabla lectura_sensores con todos los campos requeridos."""
        with conectar() as conn:
            # Activa el modo WAL para evitar el error 'database is locked'
            conn.execute('PRAGMA journal_mode=WAL;')
            
            cursor = conn.cursor()
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS lectura_sensores (
                    id INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
                    co2 INTEGER,
                    temp NUMERIC,
                    hum NUMERIC,
                    fecha TEXT,
                    lugar TEXT,
                    altura NUMERIC,
                    presion NUMERIC,
                    presion_nm NUMERIC,
                    temp_ext NUMERIC
                );
            """)
            conn.commit()
    
    # --- FUNCIONES CRUD ---
    
    def crear_sensor(datos):
        """Inserta una nueva lectura completa en la base de datos."""
        with conectar() as conn:
            cursor = conn.cursor()
            sql = """
                INSERT INTO lectura_sensores 
                (co2, temp, hum, fecha, lugar, altura, presion, presion_nm, temp_ext)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """
            parametros = (
                datos.get('co2'),
                datos.get('temp'),
                datos.get('hum'),
                datos.get('fecha'),
                datos.get('lugar'),
                datos.get('altura'),
                datos.get('presion'),
                datos.get('presion_nm'),
                datos.get('temp_ext')
            )
            cursor.execute(sql, parametros)
            return cursor.lastrowid
    
    def obtener_sensores():
        """Retorna todas las lecturas registradas."""
        with conectar() as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT * FROM lectura_sensores")
            # Convertimos a lista de diccionarios para que Flask pueda usar jsonify
            return [dict(fila) for fila in cursor.fetchall()]
    
    def obtener_sensor_por_id(id_buscado):
        """Retorna una lectura específica por su ID."""
        with conectar() as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT * FROM lectura_sensores WHERE id = ?", (id_buscado,))
            fila = cursor.fetchone()
            return dict(fila) if fila else None
    
    def actualizar_sensor(id_buscado, datos):
        """Modifica una lectura existente."""
        with conectar() as conn:
            cursor = conn.cursor()
            sql = """
                UPDATE lectura_sensores
                SET co2 = ?, temp = ?, hum = ?, fecha = ?, lugar = ?, 
                    altura = ?, presion = ?, presion_nm = ?, temp_ext = ?
                WHERE id = ?
            """
            parametros = (
                datos.get('co2'), datos.get('temp'), datos.get('hum'), 
                datos.get('fecha'), datos.get('lugar'), datos.get('altura'), 
                datos.get('presion'), datos.get('presion_nm'), datos.get('temp_ext'), 
                id_buscado
            )
            cursor.execute(sql, parametros)
            return cursor.rowcount > 0
    
    def eliminar_sensor(id_buscado):
        """Borra una lectura de la base de datos."""
        with conectar() as conn:
            cursor = conn.cursor()
            cursor.execute("DELETE FROM lectura_sensores WHERE id = ?", (id_buscado,))
            return cursor.rowcount > 0


3) Puntos 6 y 7:
Simular capturas de sensores ambientales
Generar lecturas aleatorias de CO₂, temperatura y humedad; almacenarlas en la base de datos y permitir configurar cantidad de capturas e intervalo entre mediciones.

Integrar datos externos de clima
Usar una función similar a geo_latlon() para obtener temperatura exterior, presión y humedad desde una API climática, y relacionar esos datos con las lecturas internas.


Simulacion de sensores: 
```python
    import database2 as db
    import requests
    import random
    import time
    from datetime import datetime
    
    URL = "http://127.0.0.1:5001/sensors"
    
    def simular_sensores(cantidad_capturas=10, intervalo_segundos=2):
        print(f"--- Iniciando simulación: {cantidad_capturas} capturas cada {intervalo_segundos}s ---")
        
        # Aseguramos que la tabla exista
        db.crear_tabla()
    
        for i in range(cantidad_capturas):
            # Generar datos aleatorios realistas
            lectura = {
                "co2": random.randint(400, 900),          # ppm
                "temp": round(random.uniform(18.0, 28.0), 2), # Celsius
                "hum": round(random.uniform(30.0, 60.0), 2),  # Porcentaje
                "fecha": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "lugar": "Laboratorio Central",
                "altura": 10,
                "presion": 1013,
                "presion_nm": 1015,
                "temp_ext": 15.5
            }
            
           
            
            try:
                respuesta = requests.post(URL,json=lectura)
                if respuesta.status_code == 201:
                    datos_servidor = respuesta.json()
                    id_generado = datos_servidor['sensor']['id']
                    print(f"[{i+1}/{cantidad_capturas}] Guardado ID: {id_generado} | CO2: {lectura['co2']} | Temp: {lectura['temp']}°C")
                    print(f"[{i+1}/{cantidad_capturas}] ÉXITO: Guardado ID {id_generado} | CO2: {lectura['co2']}")
                else:
                    print(f"[{i+1}/{cantidad_capturas}]  ERROR: Código {respuesta.status_code} - {respuesta.text}")
    
            except requests.exceptions.ConnectionError:
                print(f"[{i+1}/{cantidad_capturas}] ERROR")
                break 
    
    
            if i < cantidad_capturas - 1:
                time.sleep(intervalo_segundos)
    
        print("--- Simulación finalizada con éxito ---")
    
    if __name__ == "__main__":
        # Aquí puedes configurar la cantidad e intervalo
        n = int(input("¿Cuántas capturas quieres generar?: "))
        t = int(input("¿Cada cuántos segundos?: "))
        
        simular_sensores(cantidad_capturas=n, intervalo_segundos=t)



-Funcion similar a geo_latlon():

    import requests
    import geocoder
    import requests
    import database as db
    from flask import Flask, jsonify, request
    
    
    URL = "http://127.0.0.1:5001/sensors"
    
    def geo_latlon():
        g = geocoder.ip("me")
    
        if not g.latlng:
            raise Exception("No se pudo obtener la ubicación por IP")
    
        lat, lon = g.latlng
    
        url = (
            "https://api.openweathermap.org/data/2.5/weather?"
            f"lat={lat}&lon={lon}"
            f"&appid={OPENWEATHER_API_KEY}"
            "&units=metric"
            "&lang=es"
        )
    
        response = requests.get(url, timeout=10)
        data = response.json()
    
        if str(data.get("cod")) not in ["200"]:
            raise Exception(f"Error OpenWeatherMap: {data}")
    
        main = data["main"]
        weather = data["weather"][0]
    
        temp_ext = main["temp"]
        presion = main["pressure"]
        humedad_ext = main["humidity"]
        descripcion_clima = weather["description"]
    
        return {
            "lat": lat,
            "lon": lon,
            "temp_ext": temp_ext,
            "presion": presion,
            "humedad_ext": humedad_ext,
            "descripcion_clima": descripcion_clima
        }
    
    def guardarDatos():
        try:
            clima = geo_latlon()
            lectura ={
                "lugar":"Argentina",
                "altura":14,
                "presion": clima["presion"],
                "presion_nm" : 2,
                "temp_ext": clima["temp_ext"],
            }
    
            try:
                respuesta = requests.post(URL,json=lectura)
                if respuesta.status_code == 201:
                    datos_servidor = respuesta.json()
                    id_generado = datos_servidor['sensor']['id']
                    print("Exito al guardar los datos")
                else:
                    print(f" ERROR: Código {respuesta.status_code} - {respuesta.text}")
    
            except requests.exceptions.ConnectionError:
                print(f"ERROR")
        except:
            return jsonify({
                "error": "Debe enviar datos en formato JSON"
            }), 415
    
    if __name__ == "__main__":
        guardarDatos()

4) Punto 8:
   -Explicar las razones por la cual lo anterior es una Arquitectura CLiente Servidor Rest. Indicar si se cumplen los requisitos. Explicar dónde se definen el cliente y el servidor. Qué elementos faltan.


   -Este sistema implementa una Arquitectura Cliente-Servidor basada en REST porque separa claramente las responsabilidades de almacenamiento y procesamiento de datos de las interfaces que consumen dicha información, utilizando el protocolo HTTP como canal de comunicación.

   -El servidor: Es el programa que "escucha" y espera peticiones. Centraliza la lógica de negocio, gestiona la base de datos y expone EndPoints (puntos de acceso) para que los clientes interactúen con los recursos.

   -El cliente: Son los programas que inician la comunicación. El cliente no tiene acceso directo a la base de datos; en su lugar, envía solicitudes HTTP (POST, GET, etc.) al servidor para solicitar o enviar información.


-Es una Arquitectura REST ya que:

    -Uso de Métodos HTTP: tanto GET,POST,PUT,DELETE.
    -Recursos Identificables: Cada lectura de sensor se identifica de forma única mediante una URL.
    -Representación de Datos: El servidor y el cliente intercambian información en formato JSON, que es el estándar de facto para REST, asegurando que ambos hablen el mismo "idioma" independientemente del lenguaje de programación.
    -Protocolo Sin Estado (Stateless): Cada petición del cliente contiene toda la información necesaria para que el servidor la procese. El servidor no necesita "recordar" quién es el cliente entre una lectura y otra.

-Podria faltar:

    -Autenticacion: El código tiene rutas de /login que solo devuelven un mensaje de éxito, pero no protegen los recursos.
    -Validacion: El servidor asume que los datos enviados por el cliente son correctos.
    

5)          Punto 9:
  
   -Comparando este modelo con el desarrollado en el Tp1 Cliente-Servidor concurrente podemos observar ciertas similitudes y diferencias.

        -Similitudes:
    
-Arquitectura Cliente-Servidor: Ambos sistemas separan las responsabilidades; el servidor espera conexiones y el cliente inicia la petición.

-Orientados a Conexión: Ambos utilizan TCP en el fondo para asegurar que los datos lleguen sin errores y en el orden correcto.

-Capacidad Multiusuario: El primer modelo (Flask) puede atender a varios clientes, al igual que el segundo modelo mediante el uso de hilos (threading).

-Uso de Red Local: Ambos están configurados por defecto para correr en 127.0.0.1 (localhost).

            -Diferencias:
        
    -Gestion de conexion:
-REST (HTTP): Es generalmente "stateless" (sin estado). El cliente hace una petición, recibe una respuesta y la conexión se cierra (o queda en espera para otra petición independiente).
            
-Sockets: Es "stateful" (con estado). El cliente en mantiene un socket abierto con el servidor. El servidor sabe cuánto tiempo lleva conectado cada cliente (tiempo_conectado) mientras la sesión siga activa.

    -Estructura de mensajes:
-REST: Los datos se estructuran en pares clave-valor (JSON). No necesitas preocuparte por el tamaño del mensaje porque HTTP lo gestiona.

-Sockets: Tienes que definir un protocolo propio.

    -Complejidad de Implementación:
-REST: Es más fácil de escalar y mantener. Utiliza librerías de alto nivel como requests en el cliente.

-Sockets: Requiere manejar conceptos más técnicos como Endianness,lectura de archivos cada 1024 bytes, etc.
            -

6) Punto 10
   
-Desarrollar cliente visual WebSocket

-Crear una interfaz web que se conecte a un servidor WebSocket, muestre estado de conexión, permita enviar mensajes y visualice respuestas en tiempo real, según el notebook Cliente_Servidor_Websockets_R2.ipynb.

    -Servidor WS:
    
```python
    const WebSocket = require('ws');
    const wss = new WebSocket.Server({ port: 8080 });
    
    console.log("Servidor iniciado en ws://localhost:8080");
    
    wss.on('connection', (ws) => {
        console.log("Cliente conectado");
        ws.send("Conexión exitosa con el servidor");
    
        ws.on('message', (data) => {
            console.log("Recibido:", data.toString());
            // Reenvía el mensaje al cliente (Eco)
            ws.send("Servidor dice: " + data);
        });
    });


-Cliente Visual:

    from flask import Flask
    
    app = Flask(__name__)
    
    @app.route("/")
    def index():
        return """
    <!DOCTYPE html>
    <html lang="es">
    <head>
        <meta charset="UTF-8">
        <title>Panel de Control WebSocket</title>
        <style>
            body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background: #f0f2f5; padding: 40px; }
            .container { max-width: 600px; margin: auto; background: white; padding: 25px; border-radius: 15px; box-shadow: 0 10px 25px rgba(0,0,0,0.1); }
            #estado { padding: 10px; border-radius: 5px; margin-bottom: 20px; font-weight: bold; text-align: center; }
            .online { background: #d4edda; color: #155724; }
            .offline { background: #f8d7da; color: #721c24; }
            #log { height: 300px; border: 1px solid #ddd; background: #1e1e1e; color: #d4d4d4; overflow-y: scroll; padding: 10px; font-family: monospace; border-radius: 5px; }
            .input-group { margin-top: 20px; display: flex; gap: 10px; }
            input { flex-grow: 1; padding: 10px; border: 1px solid #ccc; border-radius: 5px; }
            button { padding: 10px 20px; background: #007bff; color: white; border: none; border-radius: 5px; cursor: pointer; }
            button:hover { background: #0056b3; }
            .msg-in { color: #b5cea8; } /* Color para mensajes recibidos */
            .msg-out { color: #569cd6; } /* Color para mensajes enviados */
        </style>
    </head>
    <body>
        <div class="container">
            <h2>Visual WebSocket Client</h2>
            <div id="estado" class="offline">Estado: Desconectado</div>
            
            <div id="log"></div>
    
            <div class="input-group">
                <input type="text" id="mensaje" placeholder="Escribe un mensaje..." onkeypress="if(event.key==='Enter') enviar()">
                <button onclick="enviar()">Enviar</button>
            </div>
        </div>
    
        <script>
            const logDiv = document.getElementById('log');
            const estadoDiv = document.getElementById('estado');
            const inputMsg = document.getElementById('mensaje');
    
            // 1. Iniciar conexión WebSocket
            const socket = new WebSocket('ws://localhost:8080');
    
            // Evento: Conexión abierta
            socket.onopen = () => {
                estadoDiv.innerText = "Estado: Conectado a ws://localhost:8080";
                estadoDiv.className = "online";
                addLog("SISTEMA: Conexión establecida", "yellow");
            };
    
            // Evento: Mensaje recibido
            socket.onmessage = (event) => {
                addLog("SRV: " + event.data, "msg-in");
            };
    
            // Evento: Error
            socket.onerror = (error) => {
                addLog("ERROR: No se pudo conectar al servidor", "#ff5555");
            };
    
            // Evento: Conexión cerrada
            socket.onclose = () => {
                estadoDiv.innerText = "Estado: Desconectado";
                estadoDiv.className = "offline";
                addLog("SISTEMA: Conexión cerrada", "yellow");
            };
    
            function enviar() {
                const texto = inputMsg.value;
                if (texto.trim() !== "" && socket.readyState === WebSocket.OPEN) {
                    socket.send(texto);
                    addLog("YO: " + texto, "msg-out");
                    inputMsg.value = "";
                }
            }
    
            function addLog(txt, clase) {
                const p = document.createElement('div');
                p.className = clase;
                p.style.color = (clase.includes('msg')) ? '' : clase;
                p.innerText = `[${new Date().toLocaleTimeString()}] ${txt}`;
                logDiv.appendChild(p);
                logDiv.scrollTop = logDiv.scrollHeight;
            }
        </script>
    </body>
    </html>
    """
    
    if __name__ == "__main__":
        app.run(port=5000)